# Lesson 4: transformer

Stage 4 - the full forward pass, with the shape of the data after every step.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Normansrule/transparent-transformer-llm/blob/main/notebooks/04_transformer.ipynb) &nbsp; [lesson page](../stages/04_transformer/) &nbsp;|&nbsp; [live in your browser](https://Normansrule.github.io/transparent-transformer-llm/#4)

The output below was produced by the model saved in this repository. Run the cells to reproduce it, then change things.

In [1]:
# Setup: works on GitHub Codespaces, on your own machine, and on Google Colab (where it clones the repository first).
import os, sys
if not os.path.exists("transparent_transformer"):
    if os.path.exists("../transparent_transformer"):
        os.chdir("..")
    else:
        os.system("git clone -q https://github.com/Normansrule/transparent-transformer-llm.git")
        os.chdir("transparent-transformer-llm")
sys.path.insert(0, os.getcwd())
print("ready, working in", os.getcwd())

ready, working in /home/claude/transparent-transformer-llm


In [2]:
import numpy as np

from transparent_transformer import GPT, BPETokenizer, paths
from transparent_transformer.alignment import format_prompt

tok = BPETokenizer.load(paths.TOKENIZER)
model = GPT.load(paths.ALIGNED_MODEL)
cfg = model.cfg

print("WHERE THE PARAMETERS LIVE")
groups: dict[str, int] = {}
for name, p in model.parameters().items():
    key = "embedding (token + position tables)" if name.startswith("embed") else           "attention (Q, K, V and output projections)" if ".attn." in name else           "MLP (Multi-Layer Perceptron)" if ".mlp." in name else "layer norms"
    groups[key] = groups.get(key, 0) + p.size
total = model.num_parameters()
for k, v in groups.items():
    print(f"   {k:<46}{v:>9,}  {'#' * int(40 * v / total)}")
print(f"   {'total':<46}{total:>9,}\n")

ids = np.array([tok.encode(format_prompt("What is the weather in Los Angeles?"))])
logits = model.forward(ids, capture=True)
cap = model.captured
print("THE FORWARD PASS  (B = batch, T = tokens, d = d_model, V = vocab_size)")
print(f"   token ids                     {ids.shape}            (B, T)")
print(f"   after embedding               {cap['stream'][0].shape}        (B, T, d)")
for i in range(cfg.n_layers):
    print(f"   block {i+1}: attention weights    {cap['attention'][i].shape}     (B, heads, T, T)")
    print(f"   block {i+1}: stream after block   {cap['stream'][i+1].shape}        (B, T, d)   <- same shape in, same shape out")
print(f"   logits                        {logits.shape}       (B, T, V)   one score per vocabulary entry, per position\n")

last = logits[0, -1]
print("the 5 highest-scoring next tokens after the prompt:")
for j in np.argsort(-last)[:5]:
    print(f"   {tok.token_str(j)!r:>10}  logit {last[j]:+.2f}")
print("\nLogits are raw scores, not probabilities yet. Stage 9 handles that.")

WHERE THE PARAMETERS LIVE
   embedding (token + position tables)              53,248  #############
   layer norms                                         640  
   attention (Q, K, V and output projections)       33,280  ########
   MLP (Multi-Layer Perceptron)                     66,176  #################
   total                                           153,344

THE FORWARD PASS  (B = batch, T = tokens, d = d_model, V = vocab_size)
   token ids                     (1, 11)            (B, T)
   after embedding               (1, 11, 64)        (B, T, d)
   block 1: attention weights    (1, 4, 11, 11)     (B, heads, T, T)
   block 1: stream after block   (1, 11, 64)        (B, T, d)   <- same shape in, same shape out
   block 2: attention weights    (1, 4, 11, 11)     (B, heads, T, T)
   block 2: stream after block   (1, 11, 64)        (B, T, d)   <- same shape in, same shape out
   logits                        (1, 11, 768)       (B, T, V)   one score per vocabulary entry, per position

## Your turn

**Think first:** What does each block do to the residual stream?

Then open `classroom/exercises/ex04.py`, fill in the function, and run the cell below to grade it.

In [3]:
!python classroom/check.py | head -14

## Homework: 0 of 10 passed

| exercise | lesson | result |
|---|---|---|
| `ex01.py` | Input | ⬜ not started |
| `ex02.py` | Tokenization | ⬜ not started |
| `ex03.py` | Embedding | ⬜ not started |
| `ex04.py` | Transformer | ⬜ not started |
| `ex05.py` | Attention | ⬜ not started |
| `ex06.py` | Pretraining | ⬜ not started |
| `ex07.py` | Backpropagation | ⬜ not started |
| `ex08.py` | Alignment | ⬜ not started |
| `ex09.py` | Sampling | ⬜ not started |
| `ex10.py` | Output | ⬜ not started |
Traceback (most recent call last):
  File "/home/claude/transparent-transformer-llm/classroom/check.py", line 76, in <module>
    print(summary)
BrokenPipeError: [Errno 32] Broken pipe
